# Pipeline Helper
Manual VLM testing pipeline for video analysis.

**Stages:**
1. Run global analysis → paste VLM output → saved to `<stem>.global.vlm.json`
2. Load analysis + global result → select a segment → generate segment prompt
3. Run segment analysis → paste VLM output → saved to `<start>_<end>.segment.vlm.json`

In [34]:
import hashlib
import json
import subprocess
from pathlib import Path
from typing import Any

from prompts import (
    GLOBAL_ANALYSIS_PROMPT,
    NARRATION_PROMPT,
    SEGMENT_ANALYSIS_PROMPT,
    STORY_WRITING_PROMPT,
)

## Config

In [35]:
VIDEO_FILE    = Path("../local/videos/DJI_20250514105245_0135_D.MP4")
ANALYSIS_PATH = Path("../local/videos/DJI_20250514105245_0135_D.analysis.json")
OUTPUT_DIR    = Path("../local/vlm")

## Utilities

In [36]:
def read_analysis(path: str | Path) -> dict:
    """Load an analysis.json file and return the parsed dict."""
    with open(path) as f:
        return json.load(f)


def _format_time(seconds: float) -> str:
    """Format seconds as MM:SS.mmm."""
    minutes = int(seconds) // 60
    secs = seconds - minutes * 60
    return f"{minutes:02d}:{secs:06.3f}"

In [37]:
DEFAULT_SCENE_ATTRS = ["start_time", "end_time"]


def _get_nested(obj: dict, key: str) -> Any:
    """Resolve a dotted key path like 'channel_metrics.pan.rms' from a dict."""
    for part in key.split("."):
        if not isinstance(obj, dict) or part not in obj:
            return None
        obj = obj[part]
    return obj


def extract_scene_attributes(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
) -> list[dict]:
    """
    Extract specified attributes from each scene.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    attributes:
        Scene-level keys to include. Dotted paths are supported,
        e.g. ``["start_time", "end_time", "quality_score"]``.

    Returns
    -------
    List of dicts, one per scene, always including ``scene_id``.
    """
    rows = []
    for scene in analysis.get("scenes", []):
        row = {"scene_id": scene["scene_id"]}
        for attr in attributes:
            row[attr] = _get_nested(scene, attr)
        rows.append(row)
    return rows

## VLM Prompt Generators

In [38]:
def _load_json(source: str | Path) -> dict | list:
    """Load JSON from a file path or a raw JSON string."""
    if isinstance(source, Path):
        with open(source) as f:
            return json.load(f)
    p = Path(source)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return json.loads(source)


def print_global_analysis(global_result: str | Path | dict, indent: int = 2) -> None:
    """Pretty-print the JSON output from a global video analysis."""
    data = global_result if isinstance(global_result, dict) else _load_json(global_result)
    print(json.dumps(data, indent=indent))


def extract_description(global_result: str | Path | dict) -> str:
    """
    Extract the description from a global analysis result.

    Accepts both the raw VLM output ``{"description": ...}`` and the wrapped
    format saved by ``save_global_vlm_output`` ``{"vlm": {"description": ...}}``.
    """
    data = global_result if isinstance(global_result, dict) else _load_json(global_result)
    if "vlm" in data:        # unwrap saved format
        data = data["vlm"]
    description = data.get("description")
    if not description:
        raise KeyError("'description' field not found in global analysis result.")
    return description


def _video_hex_id(video_file: str | Path) -> str:
    """Short 8-char hex ID derived from the video filename."""
    return hashlib.md5(Path(video_file).name.encode()).hexdigest()[:8]


def _extract_video_exif(video_path: str | Path) -> dict:
    """Extract video metadata via ffprobe. Returns {} if unavailable."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_streams", "-show_format",
        str(video_path),
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            return json.loads(result.stdout)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return {}

In [39]:
def generate_global_analysis_prompt(print_it=False) -> str:
    """Return the global video analysis prompt ready to send to a VLM."""
    if print_it:
        print(GLOBAL_ANALYSIS_PROMPT)
    return GLOBAL_ANALYSIS_PROMPT


def generate_segment_analysis_prompt(
    analysis: dict,
    global_result: str | Path | dict,
    segment_id: int,
    template: str | None = None,
    time_format: str = "timecode",
) -> str:
    """
    Build the VLM prompt for analysing a single segment.

    Uses ``extract_description`` to pull the global summary and
    ``extract_scene_attributes`` to compile the full segment index.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    global_result:
        Global analysis output – parsed dict, JSON string, or path to a JSON
        file produced by GLOBAL_ANALYSIS_PROMPT (both raw and wrapped formats
        are accepted).
    segment_id:
        The segment ID to analyse.
    template:
        Custom prompt template.  Defaults to SEGMENT_ANALYSIS_PROMPT.
    time_format:
        ``"timecode"`` (default, MM:SS.mmm) or ``"seconds"``.

    Returns
    -------
    The complete formatted prompt string.
    """
    tmpl = template or SEGMENT_ANALYSIS_PROMPT

    global_summary = extract_description(global_result)

    seg_attrs = extract_scene_attributes(analysis, ["start_time", "end_time"])
    seg_rows = []
    for seg in sorted(seg_attrs, key=lambda s: s["scene_id"]):
        start = seg["start_time"]
        end = seg["end_time"]
        if time_format == "timecode":
            start_str = _format_time(start)
            end_str = _format_time(end)
        else:
            start_str = f"{start:.3f}s"
            end_str = f"{end:.3f}s"
        seg_rows.append({"segment_id": seg["scene_id"], "start": start_str, "end": end_str})

    return tmpl.format(
        global_summary=global_summary,
        segments=json.dumps(seg_rows, indent=2),
        segment_id=segment_id,
    )

## VLM prompt generation

## Pipeline

In [11]:
glob_prompt = generate_global_analysis_prompt(True)

You are a Professional Video Editor and Colorist. Watch the entire video before responding.

Return a single JSON object with exactly these fields. No markdown fences, no extra keys.

{
  "description": "<Prose narrative from beginning to end. Cover setting, subjects, actions, dialogue or voiceover, visual style (color, lighting, camera work, editing rhythm), and pacing. Present tense. Minimum 3 sentences.>",

  "key_subjects": [
    ["<name>", "<one sentence: who/what this is and their role in the video>"]
  ],

  "tone": ["<3–8 adjectives describing emotional register and stylistic feel, e.g. intimate, frenetic, melancholic>"],

  "genre_or_type": "<one word or compound (underscores allowed) reflecting the video's primary intent, e.g. documentary, commercial, tutorial, music_video>",

  "tags": ["<5–10 lowercase keywords covering subject, genre, mood, and distinctive visual or audio elements>"]
}



### 1 – Global analysis output

In [12]:
def save_global_vlm_output(
    video_path: str | Path,
    vlm_json_str: str,
    output_dir: str | Path = Path("../local/vlm"),
) -> Path:
    """
    Persist the global VLM analysis output to a JSON file.

    Output: ``<output_dir>/<video_stem>/<video_stem>.global.vlm.json``

    The file contains: video_id, video_file, exif, and the VLM output
    under the key ``vlm``.

    Parameters
    ----------
    video_path:
        Path to the source video file.
    vlm_json_str:
        Raw JSON string returned by the VLM for GLOBAL_ANALYSIS_PROMPT.
    output_dir:
        Root output directory.

    Returns
    -------
    Path to the written JSON file.
    """
    video_path = Path(video_path)
    out_dir = Path(output_dir) / video_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / f"{video_path.stem}.global.vlm.json"

    record = {
        "video_id": _video_hex_id(video_path),
        "video_file": video_path.name,
        "exif": _extract_video_exif(video_path),
        "vlm": json.loads(vlm_json_str),
    }

    with open(out_file, "w") as f:
        json.dump(record, f, indent=2)

    print(f"Saved: {out_file}")
    return out_file

In [32]:
# DJI_20250514105245_0135_D
# DJI_20250801134716_0192_D
# DJI_20250802144037_0199_D
# DJI_20250802150946_0204_D
# DJI_20250803144140_0220_D
# DJI_20250803143318_0215_D
# DJI_20250731211914_0167_D
# DJI_20250803143318_0215_D

# ── 1. Paste the VLM output for the global analysis below ───────────────────
VIDEO_FILE = Path("../local/videos/DJI_20250803143318_0215_D.MP4")
ANALYSIS_PATH = Path("../local/videos/DJI_20250803143318_0215_D.analysis.json")
OUTPUT_DIR = Path("../local/vlm")

VLM_GLOBAL_JSON = """
{
"description": "An aerial drone shot captures a breathtaking mountain landscape featuring a vibrant turquoise lake nestled among steep, rocky slopes covered in patches of green vegetation. The camera smoothly pans left and lowers its altitude, tracking along a narrow dirt trail etched into the mountainside where several small figures can be seen walking. The drone eventually zeroes in on a group of three hikers, flying past them before tilting upwards to reveal the imposing, craggy mountain peaks reaching into an overcast sky. The continuous, sweeping camera movement and natural lighting emphasize the vast scale and natural beauty of the environment.",
"key_subjects": [
["Mountain landscape", "The breathtaking natural environment featuring a turquoise lake, steep trails, and rocky peaks that serves as the primary setting."],
["Hikers", "A small group of three people walking along the trail, providing a sense of scale against the vast scenery."]
],
"tone": [
"majestic",
"serene",
"expansive",
"adventurous",
"awe-inspiring"
],
"genre_or_type": "aerial_videography",
"tags": [
"drone",
"aerial",
"mountains",
"lake",
"hiking",
"nature",
"landscape",
"turquoise water",
"outdoors"
]
}
"""

# ────────────────────────────────────────────────────────────────────────────
save_global_vlm_output(VIDEO_FILE, VLM_GLOBAL_JSON, OUTPUT_DIR)

Saved: ../local/vlm/DJI_20250803143318_0215_D/DJI_20250803143318_0215_D.global.vlm.json


PosixPath('../local/vlm/DJI_20250803143318_0215_D/DJI_20250803143318_0215_D.global.vlm.json')

### 2 – Segment prompt

In [ ]:
# ── 2. Select a segment and generate its VLM prompt ─────────────────────────
# VIDEO_FILE    = Path("../local/videos/DJI_20250514105245_0135_D.MP4")
# ANALYSIS_PATH = Path("../local/videos/DJI_20250514105245_0135_D.analysis.json")
GLOBAL_RESULT = OUTPUT_DIR / VIDEO_FILE.stem / f"{VIDEO_FILE.stem}.global.vlm.json"

SEGMENT_IDX = 2  # ← change to select a different segment

# ────────────────────────────────────────────────────────────────────────────
analysis = read_analysis(ANALYSIS_PATH)

# Build segment list: [(segment_id, start_time, end_time), ...]  sorted by segment_id
segments = [
    (s["scene_id"], s["start_time"], s["end_time"])
    for s in sorted(analysis["scenes"], key=lambda s: s["scene_id"])
]

print("Available segments:")
for i, (sid, start, end) in enumerate(segments):
    marker = " ◀" if i == SEGMENT_IDX else ""
    print(f"  [{i}]  segment {sid:>2d}   {_format_time(start)} – {_format_time(end)}{marker}")

selected = segments[SEGMENT_IDX]
print(f"\nGenerating prompt for: segment {selected[0]}  "
      f"({_format_time(selected[1])} – {_format_time(selected[2])})\n")

prompt = generate_segment_analysis_prompt(analysis, GLOBAL_RESULT, selected[0])
print(prompt)

Available segments:
  [0]  segment  0   00:00.000 – 00:12.750
  [1]  segment  1   00:12.750 – 01:01.500
  [2]  segment  2   01:01.500 – 01:14.000 ◀
  [3]  segment  3   01:14.000 – 01:36.500
  [4]  segment  4   01:36.500 – 01:50.250
  [5]  segment  5   01:50.250 – 02:00.000

Generating prompt for: segment 2  (01:01.500 – 01:14.000)

You are a Professional Video Editor and Colorist performing a detailed analysis of a single scene extracted from a larger video.

### Global context
The following is a high-level summary of the full video this scene belongs to,
together with the splitting of the video in segments. Use this to understand narrative and stylistic context, but focus your analysis on what is visible in *this* clip.

<global_context>
Global Summary: The video features drone footage following a dark grey SUV as it navigates a rugged dirt road through a vast, mountainous landscape. The scenery is characterized by rolling green hills, patches of barren earth, and snow-capped peaks in

### 3 – Segment analysis output

In [22]:
def save_segment_vlm_output(
    video_path: str | Path,
    segment: tuple,
    vlm_json_str: str,
    output_dir: str | Path = Path("../local/vlm"),
) -> Path:
    """
    Persist the segment VLM analysis output to a JSON file.

    Output: ``<output_dir>/<video_stem>/<start>_<end>.segment.vlm.json``

    The file contains: segment_id, video_file, start, end, and the VLM
    output under the key ``vlm``.

    Parameters
    ----------
    video_path:
        Path to the source video file.
    segment:
        Tuple of ``(segment_id, start_time, end_time)`` as built by the
        segment selector cell.
    vlm_json_str:
        Raw JSON string returned by the VLM for SEGMENT_ANALYSIS_PROMPT.
    output_dir:
        Root output directory.

    Returns
    -------
    Path to the written JSON file.
    """
    video_path = Path(video_path)
    segment_id, start, end = segment

    out_dir = Path(output_dir) / video_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / f"{start:.3f}_{end:.3f}.segment.vlm.json"

    seg_uid = hashlib.md5(f"{video_path.name}:{start:.3f}".encode()).hexdigest()[:8]

    record = {
        "segment_id": seg_uid,
        "video_file": video_path.name,
        "start": start,
        "end": end,
        "vlm": json.loads(vlm_json_str),
    }

    with open(out_file, "w") as f:
        json.dump(record, f, indent=2)

    print(f"Saved: {out_file}")
    return out_file

In [27]:
# ── 3. Paste the VLM output for the selected segment below ──────────────────
# `selected` and `VIDEO_FILE` are carried over from cell 2 above.

VLM_SEGMENT_JSON = """
[
  {
    "segment_id": "2",
    "description": "Wide shot of a dark grey SUV driving along a dirt road through a vast, green, mountainous landscape under an overcast sky.",
    "technical_specs": {
      "framing": "Wide Shot",
      "movement": "Arc Shot",
      "angle": "High Angle",
      "reasoning": {
        "framing": "The frame captures the vehicle within its expansive natural environment, emphasizing the isolation and scale of the landscape.",
        "movement": "The camera tracks the moving vehicle while smoothly arcing around it from an elevated rear perspective to a side profile.",
        "angle": "An elevated perspective provides a clear, sweeping view of the road trajectory and the rolling terrain surrounding the vehicle."
      }
    },
    "color_profile": {
      "dominant_colors": [
        "#6a8249",
        "#a09287",
        "#9fa6ab"
      ],
      "lighting_type": "Overcast",
      "temperature": "cool"
    },
    "highlights": [
      {
        "description": "The camera completes an arc from a rear-following position to a side perspective, revealing the wider expanse of rolling green hills.",
        "keywords": [
          "arc shot",
          "landscape reveal",
          "SUV",
          "driving",
          "drone"
        ],
        "start": "00:00:05.000",
        "end": "00:00:11.000"
      }
    ],
    "quality_score": {
      "rating": "good",
      "reasoning": "The shot features smooth drone operation and steady focus, though the overcast lighting flattens the contrast and dynamic range slightly."
    },
    "segment_tags": [
      "drone footage",
      "SUV",
      "dirt road",
      "mountains",
      "overcast",
      "off-road",
      "tracking"
    ]
  }
]
"""

# ────────────────────────────────────────────────────────────────────────────
save_segment_vlm_output(VIDEO_FILE, selected, VLM_SEGMENT_JSON, OUTPUT_DIR)

Saved: ../local/vlm/DJI_20250514105245_0135_D/61.500_74.000.segment.vlm.json


PosixPath('../local/vlm/DJI_20250514105245_0135_D/61.500_74.000.segment.vlm.json')

### 4 – Story writing prompt

In [ ]:
# ── 4. Prepare the story writing prompt ─────────────────────────────────────
# Edit USER_BRIEF and VLM_SEARCH_DIR, then paste the printed prompt into the
# storyteller agent chat.
# VLM_SEARCH_DIR is scanned recursively for all *.global.vlm.json files —
# each one contributes a video-level description to the context.

USER_BRIEF      = "A cinematic travel film about off-road adventure in the mountains."  # ← edit
VLM_SEARCH_DIR  = OUTPUT_DIR  # ← directory to scan (default: same as OUTPUT_DIR)

# ─────────────────────────────────────────────────────────────────────────────
_global_files = sorted(Path(VLM_SEARCH_DIR).rglob("*.global.vlm.json"))

if not _global_files:
    print(f"No .global.vlm.json files found in: {VLM_SEARCH_DIR}")
else:
    _descriptions = []
    for _gf in _global_files:
        _data = _load_json(_gf)
        _desc = extract_description(_data)
        _video_name = _data.get("video_file", _gf.stem)
        _descriptions.append(f"[{_video_name}]\n{_desc}")

    # print(f"Found {len(_global_files)} video(s):\n" +
    #       "\n".join(f"  • {_gf}" for _gf in _global_files) + "\n")

    story_prompt = STORY_WRITING_PROMPT.format(
        user_brief=USER_BRIEF,
        video_descriptions="\n\n".join(_descriptions),
    )
    print(story_prompt)

Found 6 video(s):
  • ../local/vlm/DJI_20250514105245_0135_D/DJI_20250514105245_0135_D.global.vlm.json
  • ../local/vlm/DJI_20250731211914_0167_D/DJI_20250731211914_0167_D.global.vlm.json
  • ../local/vlm/DJI_20250801134716_0192_D/DJI_20250801134716_0192_D.global.vlm.json
  • ../local/vlm/DJI_20250802144037_0199_D/DJI_20250802144037_0199_D.global.vlm.json
  • ../local/vlm/DJI_20250803143318_0215_D/DJI_20250803143318_0215_D.global.vlm.json
  • ../local/vlm/DJI_20250803144140_0220_D/DJI_20250803144140_0220_D.global.vlm.json

You are a creative writer and storyteller. Based on the user's brief and the video descriptions provided as context, write an engaging, compelling story that captures the user's intent relying on the given video descriptions. This story is the narrative behind a video edit that will be performed using the available videos.

### User brief
A cinematic travel film about off-road adventure in the mountains.

### Video context
The following descriptions come from the ana

### 5 – Narration prompt

In [ ]:
# ── 5. Prepare the narration prompt ─────────────────────────────────────────
# Paste the story returned by the storyteller agent below, then copy the
# printed prompt into the narrator agent chat.

STORY = """
<paste story here>
"""  # ← replace with the story from step 4

# ─────────────────────────────────────────────────────────────────────────────
narration_prompt = NARRATION_PROMPT.format(story=STORY.strip())
print(narration_prompt)

## 99. Utils: Scene extraction with ffmpeg

In [20]:
def extract_scenes_to_clips(
    analysis: dict,
    output_dir: str | Path,
    video_path: str | Path | None = None,
    scene_ids: list[int] | None = None,
    scale_height: int | None = 720,
    codec: str = "h264_videotoolbox",
    ext: str = "mp4",
) -> list[Path]:
    """
    Cut a video into per-scene clips using ffmpeg, with optional downscaling.

    Output structure::

        output_dir/
        └── <video_stem>/
            ├── scene_00_720p.mp4
            ├── scene_01_720p.mp4
            └── ...

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    output_dir:
        Root folder under which a sub-folder named after the video is created.
    video_path:
        Path to the source video.  Defaults to ``analysis["video_path"]``.
    scene_ids:
        Optional list of scene IDs to extract.  Extracts all scenes when ``None``.
    scale_height:
        Output height in pixels.  Width is scaled proportionally and rounded to
        the nearest even number (``-2`` in ffmpeg terms).  Defaults to ``720``
        (720p).  Pass ``None`` to skip scaling and keep the original resolution.
    codec:
        ffmpeg video codec.  Defaults to ``"h264_videotoolbox"`` (macOS GPU
        encoder — fast).  Fall back to ``"libx264"`` on non-Apple hardware,
        or use ``"copy"`` when ``scale_height=None`` to avoid re-encoding entirely.
    ext:
        Output file extension (default ``"mp4"``).

    Returns
    -------
    List of Paths to the created clip files, sorted by scene_id.
    """
    src = Path(video_path or analysis["video_path"])
    if not src.exists():
        raise FileNotFoundError(f"Video not found: {src}")

    out_root = Path(output_dir) / src.stem
    out_root.mkdir(parents=True, exist_ok=True)

    scenes = analysis.get("scenes", [])
    if scene_ids is not None:
        scenes = [s for s in scenes if s["scene_id"] in scene_ids]
    scenes = sorted(scenes, key=lambda s: s["scene_id"])

    created = []
    for scene in scenes:
        sid = scene["scene_id"]
        start = scene["start_time"]
        end = scene["end_time"]
        duration = end - start

        suffix = f"_{scale_height}p" if scale_height else ""
        out_file = out_root / f"scene_{sid:02d}{suffix}.{ext}"

        cmd = [
            "ffmpeg", "-y",
            "-ss", str(start),
            "-i", str(src),
            "-t", str(duration),
        ]

        if scale_height:
            cmd += ["-vf", f"scale=-2:{scale_height}", "-c:v", codec, "-c:a", "copy"]
        else:
            cmd += ["-c", codec]

        cmd += ["-avoid_negative_ts", "make_zero", str(out_file)]

        res_label = f"{scale_height}p" if scale_height else "original"
        print(f"  Extracting scene {sid}  [{start:.3f}s – {end:.3f}s]  {res_label}  → {out_file}")
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"ffmpeg failed for scene {sid}:\n{result.stderr}")
        created.append(out_file)

    print(f"\nDone. {len(created)} clip(s) written to: {out_root}")
    return created

In [21]:
# ── Scene extraction ─────────────────────────────────────────────────────────
# VIDEO_FILE and ANALYSIS_PATH are defined in the Config cell above.
#
# SCALE_HEIGHT : target height in pixels (720, 1080, 480 …) or None to keep
#                original resolution (skips re-encoding entirely).
# CODEC        : "h264_videotoolbox"  → macOS GPU encoder (fast, default)
#                "libx264"            → software encoder (slower, cross-platform)
#                "copy"               → only valid when SCALE_HEIGHT is None

SCENE_IDS    = None  # ← e.g. [0, 1, 2] to extract specific scenes only
SCALE_HEIGHT = 720   # ← e.g. 1080, 720, 480, or None for original resolution

# ─────────────────────────────────────────────────────────────────────────────
analysis = read_analysis(ANALYSIS_PATH)

clips = extract_scenes_to_clips(
    analysis=analysis,
    output_dir=OUTPUT_DIR,
    video_path=VIDEO_FILE,
    scene_ids=SCENE_IDS,
    scale_height=SCALE_HEIGHT,
    codec="copy" if SCALE_HEIGHT is None else "h264_videotoolbox",
)

  Extracting scene 0  [0.000s – 12.750s]  720p  → ../local/vlm/DJI_20250514105245_0135_D/scene_00_720p.mp4
  Extracting scene 1  [12.750s – 61.500s]  720p  → ../local/vlm/DJI_20250514105245_0135_D/scene_01_720p.mp4
  Extracting scene 2  [61.500s – 74.000s]  720p  → ../local/vlm/DJI_20250514105245_0135_D/scene_02_720p.mp4
  Extracting scene 3  [74.000s – 96.500s]  720p  → ../local/vlm/DJI_20250514105245_0135_D/scene_03_720p.mp4
  Extracting scene 4  [96.500s – 110.250s]  720p  → ../local/vlm/DJI_20250514105245_0135_D/scene_04_720p.mp4
  Extracting scene 5  [110.250s – 120.000s]  720p  → ../local/vlm/DJI_20250514105245_0135_D/scene_05_720p.mp4

Done. 6 clip(s) written to: ../local/vlm/DJI_20250514105245_0135_D
